In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_absolute_error

from lightgbm import LGBMRegressor

In [2]:
df = pd.read_csv("../data/processed/features_v2.csv")

node_features = pd.read_csv(
    "../data/processed/node_features.csv"
)

In [3]:
source_features = node_features.copy()

source_features.columns = [
    "source_center",
    "source_betweenness",
    "source_pagerank",
    "source_in_degree",
    "source_out_degree"
]

df = df.merge(
    source_features,
    on="source_center",
    how="left"
)

In [4]:
dest_features = node_features.copy()

dest_features.columns = [
    "destination_center",
    "destination_betweenness",
    "destination_pagerank",
    "destination_in_degree",
    "destination_out_degree"
]

df = df.merge(
    dest_features,
    on="destination_center",
    how="left"
)

In [5]:
features = [

    "osrm_time",
    "osrm_distance",
    "actual_distance_to_destination",
    "cutoff_factor",

    "route_type",

    "corridor_median_factor",
    "corridor_trip_count",
    "corridor_p90_factor",

    "hub_outbound_delay",
    "hub_trip_volume",
    "hub_inbound_delay",

    "source_betweenness",
    "source_pagerank",
    "source_out_degree",

    "destination_betweenness",
    "destination_pagerank",
    "destination_in_degree"
]

target = "actual_time"

In [6]:
X = df[features]

X = X.fillna(0)

y = df[target]

groups = df["trip_uuid"]

In [9]:
df["route_type"] = df["route_type"].map({
    "FTL": 1,
    "Carting": 0
}).fillna(df["route_type"])

df["route_type"] = pd.to_numeric(
    df["route_type"],
    errors="coerce"
)

In [11]:
print(X.dtypes)

osrm_time                         float64
osrm_distance                     float64
actual_distance_to_destination    float64
cutoff_factor                       int64
route_type                            str
corridor_median_factor            float64
corridor_trip_count                 int64
corridor_p90_factor               float64
hub_outbound_delay                float64
hub_trip_volume                     int64
hub_inbound_delay                 float64
source_betweenness                float64
source_pagerank                   float64
source_out_degree                   int64
destination_betweenness           float64
destination_pagerank              float64
destination_in_degree               int64
dtype: object


In [12]:
bad_cols = X.select_dtypes(include=["object"]).columns
print(bad_cols)

Index(['route_type'], dtype='str')


C:\Users\sorou\AppData\Local\Temp\ipykernel_31164\2864843967.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  bad_cols = X.select_dtypes(include=["object"]).columns


In [7]:
gkf = GroupKFold(n_splits=5)

scores = []

In [14]:
for train_idx, val_idx in gkf.split(X, y, groups):

    X_train = X.iloc[train_idx]
    X_val = X.iloc[val_idx]

    y_train = y.iloc[train_idx]
    y_val = y.iloc[val_idx]

    model = LGBMRegressor(
        n_estimators=200,
        learning_rate=0.05,
        num_leaves=31,
        random_state=42
    )

    model.fit(X_train, y_train)

    preds = model.predict(X_val)

    mae = mean_absolute_error(
        y_val,
        preds
    )

    scores.append(mae)

    print(
        "Fold MAE:",
        round(mae, 2)
    )

ValueError: pandas dtypes must be int, float or bool.
Fields with bad pandas dtypes: route_type: str